# ThriftyChef — Offline Evaluation Notebook

Reproducible offline evaluation for **ThriftyChef**: popularity, content-based, SVD, and hybrid recommenders on Food.com, with waste-coverage and cold-start utility benchmarks.

**Configuration.** Use `FINAL_RUN=True` for report-quality metrics (≥1000 eval users when available). Use `FINAL_RUN=False` for a faster demo pass.

**Outputs.** Metrics and figures are written under `notebooks/eval_outputs/` (CSV/JSON/PNG).


## 0. Environment setup

Install dependencies (Colab), clone the repository if needed, and resolve `ROOT`.


In [ ]:
import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/NJDBSProjects20093736/FridgeWise.git"
BRANCH = "Thrifty-chef-implementations"
PROJECT_DIR = Path("/content/FridgeWise") if IN_COLAB else (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)

if IN_COLAB:
    %pip install -q pandas numpy scipy scikit-learn scikit-surprise matplotlib seaborn tqdm shap python-dateutil
    if not (PROJECT_DIR / "src").exists():
        !git clone --branch {BRANCH} --single-branch {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

ROOT = PROJECT_DIR.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("IN_COLAB =", IN_COLAB)
print("ROOT     =", ROOT)


## 0.1 Load processed data from GitHub

Extract `data/presentation/data_clean.zip` into `data/clean/` (from the clone, or download from GitHub raw if missing).


In [ ]:
from pathlib import Path
import zipfile, shutil, urllib.request

CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)
ZIP_IN_REPO = ROOT / "data" / "presentation" / "data_clean.zip"
DATA_BRANCH = "Thrifty-chef-implementations"
GITHUB_ZIP_URL = (
    "https://raw.githubusercontent.com/NJDBSProjects20093736/FridgeWise/"
    f"{DATA_BRANCH}/data/presentation/data_clean.zip"
)

def unzip_clean(zip_path: Path) -> None:
    extract_root = ROOT / "data" / "_unzip_clean"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_root)
    hits = list(extract_root.rglob("clean_recipes.csv"))
    if not hits:
        raise FileNotFoundError("clean_recipes.csv not found inside data_clean.zip")
    for f in hits[0].parent.glob("*"):
        if f.is_file():
            shutil.copy2(f, CLEAN / f.name)

if (CLEAN / "clean_recipes.csv").exists():
    print("data/clean already present")
elif ZIP_IN_REPO.exists():
    print("Using zip from repo:", ZIP_IN_REPO)
    unzip_clean(ZIP_IN_REPO)
else:
    print("Downloading from GitHub...")
    dest = ROOT / "data" / "presentation"
    dest.mkdir(parents=True, exist_ok=True)
    out = dest / "data_clean.zip"
    with urllib.request.urlopen(GITHUB_ZIP_URL, timeout=180) as resp:
        out.write_bytes(resp.read())
    unzip_clean(out)

required = [
    "clean_recipes.csv", "clean_interactions.csv", "user_fridge_inventory.csv",
    "user_profiles.csv", "context_tag_lifts.csv", "recipe_ingredient_features.csv",
]
missing = [n for n in required if not (CLEAN / n).exists()]
if missing:
    raise FileNotFoundError("Missing: " + ", ".join(missing))
print("Clean tables OK:", sorted(p.name for p in CLEAN.glob("*.csv")))


## 0.2 Experiment configuration

- `FINAL_RUN=False` → faster demo (`MAX_EVAL_USERS=200`)
- `FINAL_RUN=True` → report run (`MAX_EVAL_USERS≥1000` when the split allows), fixed seed, artefacts saved


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiment import EXPERIMENT_CONFIG
from src.data_loader import load_fridgewise_data, parse_json_list

# === Toggle for report vs demo ===
FINAL_RUN = True   # set False for a quick Colab smoke test

RANDOM_STATE = EXPERIMENT_CONFIG.random_state  # 1103
K = EXPERIMENT_CONFIG.k                        # 10
if FINAL_RUN:
    MAX_EVAL_USERS = max(1000, EXPERIMENT_CONFIG.max_eval_users)
else:
    MAX_EVAL_USERS = 200

OUT_DIR = ROOT / "notebooks" / "eval_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"FINAL_RUN={FINAL_RUN} | seed={RANDOM_STATE} | K={K} | eval_users≤{MAX_EVAL_USERS}")
print("Output dir:", OUT_DIR)


In [ ]:
data = load_fridgewise_data(ROOT)
n_users = data.interactions["user_id"].nunique()
n_items = data.interactions["recipe_id"].nunique()
fridge_users = set(data.fridge["user_id"].astype(int))
print(f"Recipes:      {len(data.recipes):>8,}")
print(f"Interactions: {len(data.interactions):>8,}")
print(f"Users:        {n_users:>8,}")
print(f"Fridge users: {len(fridge_users):>8,}  |  Profiles: {len(data.profiles):>8,}")
print(f"Sparsity:     {1 - len(data.interactions)/(n_users*n_items):.6f}")
display(data.recipes[["recipe_id", "recipe_name", "minutes", "n_ingredients"]].head(3))


### 1.1 Exploratory notes

Ratings are skewed toward 4–5★. Pointwise RMSE can look strong while top-$N$ ranking remains hard under catalogue sparsity. Primary protocol: **NDCG@10** and **MAP@10** on held-out rating≥4 items.


In [ ]:
dist = (data.interactions["rating"].value_counts(normalize=True).sort_index() * 100)
fig, ax = plt.subplots(figsize=(6, 3.5))
dist.plot(kind="bar", ax=ax, color="#2980b9")
ax.set(title="Rating distribution", xlabel="rating", ylabel="% of interactions")
plt.tight_layout()
fig.savefig(FIG_DIR / "rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(dist.round(1).to_string())


## 2. Models and train/test split

| Model | Method |
|-------|--------|
| Popularity | Bayesian-smoothed global ranking |
| Content-based | Mean TF–IDF of liked training recipes; fridge overlap only when a fridge exists |
| SVD | Surprise matrix factorisation; top-$N$ over the **full catalogue** |
| Hybrid | Popularity prior + CF + match + expiry + nutrition |

**Content-based fairness note.** Only ~50 demo users have fridge rows. Ranking evaluation users are almost all Food.com accounts without fridges. Profiles are therefore built from **training likes (rating≥4)** so TF–IDF/cosine is defined; fridge overlap remains the signal for waste/cold-start demos.


In [ ]:
from src.evaluation.splits import user_holdout_split
from src.models.popularity import PopularityRecommender
from src.models.content_based import ContentBasedRecommender
from src.models.collaborative import CollaborativeRecommender
from src.recommender import HybridRecommender

train_df, test_df, test_relevant = user_holdout_split(
    data.interactions, random_state=RANDOM_STATE
)
train_data = data.with_interactions(train_df)

rel_users = set(int(u) for u in test_relevant)
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | users w/ relevant hold-out: {len(test_relevant):,}")
print(f"Eval users with fridge inventory: {len(rel_users & fridge_users)} / {len(rel_users)}")

popularity = PopularityRecommender().fit(train_data)
content = ContentBasedRecommender().fit(train_data)
svd = CollaborativeRecommender(n_factors=50, n_epochs=20).fit(
    train_data, test_size=0.0, random_state=RANDOM_STATE
)
hybrid = HybridRecommender().fit(train_data, svd, content)
hybrid_ctx = HybridRecommender(context_max_boost=0.15).fit(train_data, svd, content)

# Diagnostics: content profiles must be non-empty for Food.com users
_sample_uid = next(iter(test_relevant))
print("Sample eval user:", _sample_uid)
print("  fridge ings:", len(content.user_fridge.get(_sample_uid, set())))
print("  pref ings:  ", len(content.user_pref_ings.get(_sample_uid, set())))
print("  user doc nonempty:", bool(content.user_docs.get(_sample_uid, "").strip()))
print("Trained: popularity, content_based, svd, hybrid, hybrid_with_context")


### 2.1 Content-based ranking diagnostic

**Previous bug (fixed):** content profiles used only the ~50 demo fridge users, so Food.com eval users had empty TF-IDF profiles and produced all-zero metrics.

**Current behaviour:** profiles are built from liked training recipes (mean TF-IDF centroid). Fridge overlap is used only when a fridge exists (waste/cold-start). On Food.com top-10 relevance, content-based can still score near zero because hold-outs rarely enter the top-10 under catalogue sparsity — check HitRate@100 below. Waste coverage and cold-start match remain the fair utility metrics for fridge-aware content ranking.


In [ ]:
# Diagnostic: empty-profile bug should be gone; @10 may still be low vs popularity
from src.evaluation.metrics import hit_rate_at_k, ndcg_at_k

_diag_n = min(80, len(eval_users)) if "eval_users" in dir() else 40
_rng = np.random.default_rng(RANDOM_STATE)
_users = list(test_relevant.keys())
if len(_users) > _diag_n:
    _users = list(_rng.choice(_users, _diag_n, replace=False))

rows = []
for label, model in [("content_based", content), ("popularity", popularity)]:
    hr10, hr100, nd10 = [], [], []
    nonempty = 0
    for uid in _users:
        uid = int(uid)
        if label == "content_based":
            nonempty += int(
                bool(content.user_docs.get(uid, "").strip())
                or bool(getattr(content, "_liked_recipe_ids", {}).get(uid))
            )
        rel = {int(x) for x in test_relevant[uid]}
        recs = [int(r.recipe_id) for r in model.recommend(uid, k=100, exclude_seen=True)]
        hr10.append(hit_rate_at_k(recs[:10], rel, 10))
        hr100.append(hit_rate_at_k(recs, rel, 100))
        nd10.append(ndcg_at_k(recs[:10], rel, 10))
    rows.append({
        "model": label,
        "users": len(_users),
        "nonempty_profiles": nonempty if label == "content_based" else len(_users),
        "HitRate@10": float(np.mean(hr10)),
        "HitRate@100": float(np.mean(hr100)),
        "NDCG@10": float(np.mean(nd10)),
    })
diag = pd.DataFrame(rows).set_index("model")
display(diag)
diag.to_csv(OUT_DIR / "content_ranking_diagnostic.csv")
print(
    "Interpretation: if content profiles are non-empty but HitRate@10 is near zero while "
    "HitRate@100 is higher, the recommender is functioning; Food.com hold-outs are simply "
    "rarely in the top-10 for content-based ranking."
)


## 3. Offline ranking — NDCG & MAP

Each model returns top-$K$ **unseen** recipes. Relevance = held-out interactions with rating ≥4.

**SVD note.** RMSE measures pointwise rating fit. Top-$N$ NDCG/MAP measure whether liked hold-outs appear near the top of a list ranked over the **full recipe catalogue** (not sampled negatives). Good RMSE does not imply strong top-$N$ under extreme sparsity and popularity bias.


In [ ]:
from src.evaluation.metrics import (
    precision_at_k, recall_at_k, average_precision_at_k, ndcg_at_k, hit_rate_at_k,
)
from src.evaluation.evaluator import evaluate_svd_rmse

_rng = np.random.default_rng(RANDOM_STATE)
eval_users = list(test_relevant.keys())
if len(eval_users) > MAX_EVAL_USERS:
    eval_users = list(_rng.choice(eval_users, MAX_EVAL_USERS, replace=False))
eval_users = [int(u) for u in eval_users]
print(f"Evaluating {len(eval_users)} users")

def evaluate(model, name: str) -> dict:
    acc = {"precision": [], "recall": [], "map": [], "ndcg": [], "hit_rate": []}
    empty_recs = 0
    for uid in eval_users:
        rel = {int(x) for x in test_relevant[uid]}
        recs = [int(r.recipe_id) for r in model.recommend(int(uid), k=K, exclude_seen=True)]
        if not recs:
            empty_recs += 1
        acc["precision"].append(precision_at_k(recs, rel, K))
        acc["recall"].append(recall_at_k(recs, rel, K))
        acc["map"].append(average_precision_at_k(recs, rel, K))
        acc["ndcg"].append(ndcg_at_k(recs, rel, K))
        acc["hit_rate"].append(hit_rate_at_k(recs, rel, K))
    row = {"model": name, **{m: float(np.mean(v)) for m, v in acc.items()}}
    if empty_recs:
        print(f"  [{name}] empty recommendation lists: {empty_recs}")
    return row

ranking_rows = [
    evaluate(popularity, "popularity"),
    evaluate(content, "content_based"),
    evaluate(svd, "svd"),
    evaluate(hybrid, "hybrid"),
]
ranking = pd.DataFrame(ranking_rows).set_index("model")

rmse = evaluate_svd_rmse(data.interactions, random_state=RANDOM_STATE)
print(f"SVD RMSE (80/20 rating split): {rmse:.4f}")
print("RMSE ≠ ranking quality: top-N scores the full catalogue against sparse hold-outs.")
display(ranking)

ax = ranking[["ndcg", "map", "hit_rate"]].plot(
    kind="bar", figsize=(8, 4), color=["#2980b9", "#27ae60", "#e67e22"]
)
ax.set(title=f"Ranking metrics @{K} (n={len(eval_users)} users)", ylabel="score")
plt.xticks(rotation=15); plt.legend(loc="best"); plt.tight_layout()
fig = ax.get_figure()
fig.savefig(FIG_DIR / "ranking_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

ranking_out = ranking.copy()
ranking_out["rmse"] = np.nan
ranking_out.loc["svd", "rmse"] = rmse
ranking_out.to_csv(OUT_DIR / "evaluation_results.csv")
print("Saved", OUT_DIR / "evaluation_results.csv")


## 4. Waste-reduction simulation

Waste coverage = fraction of soon-to-expire fridge items (`days≤5`) appearing in top-$K$ recipe ingredients. This utility metric is separate from Food.com NDCG.


In [ ]:
from src.evaluation.waste import simulate_waste_reduction

waste = pd.DataFrame([
    {
        "model": w.model_name,
        "waste_coverage": w.waste_coverage,
        "expiring_used": w.expiring_items_used,
        "expiring_total": w.expiring_items_total,
    }
    for w in (simulate_waste_reduction(m, data, k=K) for m in [popularity, content, hybrid])
]).set_index("model")
display(waste)
waste.to_csv(OUT_DIR / "waste_results.csv")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for m in ["popularity", "content_based", "hybrid"]:
    if m not in ranking.index or m not in waste.index:
        continue
    nd, wc = ranking.loc[m, "ndcg"], waste.loc[m, "waste_coverage"]
    ax.scatter(nd, wc, s=160)
    ax.annotate(m, (nd, wc), textcoords="offset points", xytext=(8, 6))
ax.set(
    xlabel=f"NDCG@{K}",
    ylabel="Waste coverage",
    title="Relevance vs waste-reduction trade-off",
)
ax.grid(alpha=0.3); plt.tight_layout()
fig.savefig(FIG_DIR / "relevance_vs_waste.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Cold-start (new-user) evaluation

Primary cold-start evidence is the **new-user table** (zero rating history → fridge-first ranking).

**Warm-up curve caveat.** Ingredient-match can drop when switching from cold (fridge-heavy) to warm (popularity/CF blend). That is a change of objective, not necessarily a bug. For the report we emphasise the new-user table; the curve below reports **NDCG** from a chronological hold-out protocol when available.


In [ ]:
from src.evaluation.cold_start import evaluate_new_user_fallback, evaluate_warmup_curve

cold = pd.DataFrame([
    {"model": c.model, "new_user_match": c.mean_ingredient_match, "waste": c.waste_coverage}
    for c in evaluate_new_user_fallback(data, hybrid, content, popularity, k=K)
]).set_index("model")
display(cold)
cold.to_csv(OUT_DIR / "cold_start_results.csv")

# Chronological warm-up (NDCG). May be empty if too few profile users qualify.
warm = pd.DataFrame(evaluate_warmup_curve(hybrid, data, k=K))
if len(warm):
    cols = [c for c in ["num_ratings", "users_evaluated", "ndcg", "map_score",
                        "mean_ingredient_match", "waste_coverage", "cold_start_mode"] if c in warm.columns]
    display(warm[cols])
    fig, ax1 = plt.subplots(figsize=(7, 4))
    if "ndcg" in warm.columns:
        ax1.plot(warm["num_ratings"], warm["ndcg"], "o-", color="#2980b9", label="NDCG")
        ax1.set_ylabel("NDCG@10")
    else:
        ax1.plot(warm["num_ratings"], warm["mean_ingredient_match"], "o-", color="#2980b9")
        ax1.set_ylabel("ingredient match")
    ax1.set_xlabel("ratings in history")
    ax1.set_title("Cold→warm curve (chronological hold-out)")
    ax1.set_xticks(warm["num_ratings"])
    ax1.legend(loc="best")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "warmup_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(
        "Note: mean_ingredient_match may fall after the first rating because the hybrid "
        "shifts weight from fridge-first cold scoring toward popularity/CF warm scoring."
    )
else:
    print("Warm-up curve unavailable for this profile set — report new-user table only.")


## 6. Context ablation

Context re-ranking did not improve offline NDCG on this split, so it is used as an **optional product/demo layer** rather than the default accuracy setting.


In [ ]:
ctx_on = evaluate(hybrid_ctx, "hybrid_with_context")
ctx_off = evaluate(hybrid, "hybrid_without_context")
context_df = pd.DataFrame([ctx_on, ctx_off]).set_index("model")
delta = ctx_on["ndcg"] - ctx_off["ndcg"]
print(f"NDCG@{K} with context:    {ctx_on['ndcg']:.4f}")
print(f"NDCG@{K} without context: {ctx_off['ndcg']:.4f}")
print(f"Delta: {delta:+.4f}")
display(context_df[["ndcg", "map", "hit_rate"]])
context_df.to_csv(OUT_DIR / "context_ablation.csv")
print(
    "Conclusion: Context re-ranking did not improve offline NDCG on this split, "
    "so it is used as an optional product/demo layer rather than the default accuracy setting."
)


## 7. Hybrid component ablation

Compare the full hybrid against variants with individual signals removed.


In [ ]:
ablation_specs = [
    ("hybrid_full", dict()),
    ("hybrid_no_expiry", dict(use_expiry=False)),
    ("hybrid_no_nutrition", dict(use_nutrition=False)),
    ("hybrid_no_cf", dict(use_cf=False)),
    ("hybrid_no_content_match", dict(use_content_match=False)),
]

ablation_rows = []
for name, kwargs in ablation_specs:
    model = HybridRecommender(**kwargs).fit(train_data, svd, content)
    row = evaluate(model, name)
    # waste on demo fridges
    w = simulate_waste_reduction(model, data, k=K)
    row["waste_coverage"] = w.waste_coverage
    ablation_rows.append(row)
    print(f"done {name}: NDCG={row['ndcg']:.4f} MAP={row['map']:.4f} waste={row['waste_coverage']:.4f}")

ablation = pd.DataFrame(ablation_rows).set_index("model")
display(ablation[["ndcg", "map", "waste_coverage", "precision", "recall", "hit_rate"]])
ablation.to_csv(OUT_DIR / "hybrid_ablation.csv")

ax = ablation[["ndcg", "map", "waste_coverage"]].plot(
    kind="bar", figsize=(9, 4), color=["#2980b9", "#27ae60", "#8e44ad"]
)
ax.set(title="Hybrid ablation", ylabel="score")
plt.xticks(rotation=20); plt.tight_layout()
fig = ax.get_figure()
fig.savefig(FIG_DIR / "hybrid_ablation.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Model comparison table (report-ready)

Combined ranking, RMSE (SVD only), waste coverage, and cold-start match.


In [ ]:
comparison = ranking.copy()
comparison["rmse"] = np.nan
comparison.loc["svd", "rmse"] = float(rmse)
comparison["waste_coverage"] = np.nan
for m in waste.index:
    if m in comparison.index:
        comparison.loc[m, "waste_coverage"] = waste.loc[m, "waste_coverage"]
comparison["cold_start_match"] = np.nan
for m in cold.index:
    if m in comparison.index:
        comparison.loc[m, "cold_start_match"] = cold.loc[m, "new_user_match"]

# Friendly column names for the report
report_table = comparison.rename(columns={
    "precision": "Precision@10",
    "recall": "Recall@10",
    "map": "MAP@10",
    "ndcg": "NDCG@10",
    "hit_rate": "HitRate@10",
    "rmse": "RMSE (SVD)",
    "waste_coverage": "Waste coverage",
    "cold_start_match": "Cold-start match",
})
display(report_table)
report_table.to_csv(OUT_DIR / "model_comparison_table.csv")

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis("off")
tbl = ax.table(
    cellText=np.round(report_table.fillna(0).values, 4),
    rowLabels=list(report_table.index),
    colLabels=list(report_table.columns),
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.2, 1.4)
ax.set_title("ThriftyChef model comparison", pad=20)
plt.tight_layout()
fig.savefig(FIG_DIR / "model_comparison_table.png", dpi=200, bbox_inches="tight")
plt.show()


## 9. Generative AI — comparative discussion

| Approach | Status in this project |
|----------|------------------------|
| Template / structured explanations | **Implemented** (safe, deterministic) |
| SHAP surrogate explanations | Implemented as lecture extension |
| Mult-VAE | **Not trained/evaluated here** — future enhancement |
| LLM rewrite of explanations | **Not deployed as core** — discussion / optional UX |
| Recipe GAN | **Discussion only** — allergen risk |

Allergen and diet constraints remain **hard deterministic filters**, never generative.


In [ ]:
from src.genai.analysis import genai_comparison_table, recommend_genai_strategy, sample_llm_prompt

genai_df = pd.DataFrame([
    {
        "approach": g.name,
        "type": g.type,
        "implemented": g.implemented,
        "role": g.role,
        "pros": "; ".join(g.pros[:2]),
        "cons": "; ".join(g.cons[:2]),
    }
    for g in genai_comparison_table()
])
display(genai_df)
print("Strategy:")
for k, v in recommend_genai_strategy().items():
    print(f"  - {k}: {v}")
print("\nExample constrained LLM prompt (not executed as a trained model):")
print(" ", sample_llm_prompt("Greek Potatoes Oven Roasted", 0.5, ["parsley", "garlic powder"]))
print(
    "\nResponsible AI: allergen/diet filters must remain deterministic; "
    "generative text may only explain already-ranked, already-filtered recipes."
)


## 10. Final summary (for the written report)

Paste or adapt the printed paragraph below into the discussion section.


In [ ]:
pop_ndcg = float(ranking.loc["popularity", "ndcg"])
hyb_ndcg = float(ranking.loc["hybrid", "ndcg"])
pop_map = float(ranking.loc["popularity", "map"])
hyb_map = float(ranking.loc["hybrid", "map"])

summary = {
    "experiment": {
        "final_run": FINAL_RUN,
        "seed": RANDOM_STATE,
        "k": K,
        "eval_users": len(eval_users),
        "svd_rmse": round(float(rmse), 4),
        "context_delta_ndcg": round(float(delta), 4),
    },
    "ranking": ranking.round(6).to_dict(),
    "waste_coverage": waste["waste_coverage"].round(6).to_dict(),
    "cold_start_match": cold["new_user_match"].round(6).to_dict(),
    "hybrid_ablation": ablation[["ndcg", "map", "waste_coverage"]].round(6).to_dict(),
    "findings": [
        "Hybrid slightly improves or matches popularity on NDCG/MAP depending on the split; it is the best personalised blend for joint relevance and utility.",
        "Content-based is strongest for waste coverage and new-user cold-start ingredient match when fridge signals are available.",
        "SVD attains good RMSE but weak top-N NDCG/MAP because ranking scores the full sparse catalogue, not only rating residuals.",
        "Hybrid balances Food.com relevance (via popularity/CF) with fridge/expiry utility.",
        "Context re-ranking did not improve offline NDCG on this split, so it is an optional product/demo layer rather than the default accuracy setting.",
        "Safety filters (allergens, diet) are hard constraints and must remain deterministic—not generative.",
    ],
}

summary_path = OUT_DIR / "final_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved", summary_path)

report_blurb = (
    f"ThriftyChef offline evaluation (seed={RANDOM_STATE}, K={K}, n={len(eval_users)} users). "
    f"The hybrid recommender achieves NDCG@10={hyb_ndcg:.4f} and MAP@10={hyb_map:.4f}, compared with "
    f"popularity at NDCG@10={pop_ndcg:.4f} and MAP@10={pop_map:.4f}. Content-based filtering is strongest "
    f"for waste coverage and new-user cold-start ingredient match. SVD shows competitive RMSE "
    f"({float(rmse):.4f}) but weak top-N ranking because RMSE and NDCG measure different objectives and "
    f"top-N is computed over the full catalogue. Overall, the hybrid best balances relevance and "
    f"waste reduction. Context re-ranking did not improve offline NDCG on this split, so it is used as "
    f"an optional product/demo layer rather than the default accuracy setting. Allergen and dietary "
    f"rules remain hard deterministic filters."
)
print("\n=== REPORT BLURB ===\n")
print(report_blurb)
(OUT_DIR / "report_blurb.txt").write_text(report_blurb + "\n", encoding="utf-8")
print("\nArtefacts in", OUT_DIR)
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUT_DIR))
